# Prepare Beauty Atomic Files


In [1]:
from pathlib import Path
import pandas as pd


In [ ]:
# --- Config ---
BEAUTY_CATEGORY = "Beauty_and_Personal_Care"
DATA_DIR: str = "../data"

# Data split cutoff dates
TRAIN_END_CUTOFF_DATE: str = "2022-08-01"
VALID_END_CUTOFF_DATE: str = "2022-10-01"

# User and item thresholds
USER_MIN_REVIEWS: int = 5
WARM_USER_MIN_REVIEWS: int = 10
WARM_ITEM_MIN_REVIEWS: int = 5

# Downsampling for faster iteration
RANDOM_SEED: int = 42
MAX_TRAIN_SIZE: int | None = None
MAX_VALID_SIZE: int | None = None
MAX_TEST_SIZE: int | None = None


## Load reviews


In [3]:
df_reviews = pd.read_csv(f"{DATA_DIR}/reviews.csv")
display(df_reviews.sample(10))
df_reviews.info()


,user_id,parent_asin,rating,timestamp,category
13341697,AE4BMEEZQMSJAHQHBFKIBDKG5KQA,B07XQ9WX45,5.0,1639632626860,Clothing_Shoes_and_Jewelry
16566340,AFLGFXSXFOVZZGKYCEMA2JV27VNA,B073H3GF9H,5.0,1647895868324,Clothing_Shoes_and_Jewelry
2177710,AGZWEOJWHUT4CG5TXHVQGE3OIE4A,B08FMSJH3M,5.0,1614097707843,Clothing_Shoes_and_Jewelry
25643374,AETKSDQF4Z4VHKNAZGFUL62HUOEQ,B0C34R4YFM,5.0,1669836384870,Clothing_Shoes_and_Jewelry
12229780,AGYEKZBCS6Y2PE552N4CECXUO6HA,B09P2P1KG1,5.0,1636463217668,Clothing_Shoes_and_Jewelry
25200994,AGWN7UHRAPEN45GPU4MWFM2DTRGQ,B077BPY8QC,4.0,1668698989045,Clothing_Shoes_and_Jewelry
23019631,AH4BIIZWZ3UPYEFSTRCHLMTWNYBQ,B07S4G1B3M,5.0,1663771777145,Beauty_and_Personal_Care
11789976,AHOXZDGK6RKA5LDGDZ3ZC4MJW3FA,B08QCFNBJY,5.0,1635250664075,Clothing_Shoes_and_Jewelry
10070705,AGKZUJTSIYY3XR2IM2J2HQNTYY3Q,B01EHELYU4,1.0,1630357899233,Clothing_Shoes_and_Jewelry
3838912,AHRYRSLF2LW7YCXSRG2EPL2LLLTA,B074W75SWB,2.0,1617260007128,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26876611 entries, 0 to 26876610
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      object 
 1   parent_asin  object 
 2   rating       float64
 3   timestamp    int64  
 4   category     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 1.0+ GB


## Load items


In [4]:
df_items = pd.read_csv(f"{DATA_DIR}/items.csv")
display(df_items.sample(10))
df_items.info()


,parent_asin,title,price,store,category
1917668,B0BRCWL21Z,Gleyemor Hippie Style Mushroom Shaped Sunglass...,12.99,Gleyemor,Clothing_Shoes_and_Jewelry
1377819,B06XK1KRVS,UNN Led Light Up Shoes for Men Women USB Charg...,39.99,UNN,Clothing_Shoes_and_Jewelry
771543,B08H85NCFN,NSQTBA Womens Long Sleeve V Neck Tunic Tops Lo...,NaN,NSQTBA,Clothing_Shoes_and_Jewelry
569530,B0BCVYMBNQ,SATINIOR 2 Pairs Women's Floral Lace Gloves UV...,8.58,SATINIOR,Clothing_Shoes_and_Jewelry
1598145,B08XYJN73H,Tomilee Water Shoes Quick Dry Beach Aqua Sport...,NaN,Tomilee,Clothing_Shoes_and_Jewelry
518642,B07MNGSWZJ,COOFANDY Men's Waterproof Rain Jacket with Hoo...,34.99,COOFANDY,Clothing_Shoes_and_Jewelry
2067904,B00JZYVUGU,1 Carat Solitaire Diamond Stud Earrings Round ...,1489.00,Houston Diamond District,Clothing_Shoes_and_Jewelry
2470584,B082F97YQQ,Kendra Scott Faceted Danielle Statement Earrin...,NaN,Kendra Scott,Clothing_Shoes_and_Jewelry
681970,B07XQP78WM,TOOLOUD Pho King Awesome Funny Vietnamese Soup...,19.97,TOOLOUD,Clothing_Shoes_and_Jewelry
293222,B087QC2HNC,QTHAIR 12a Grade Brazilian Deep Wave Human Hai...,NaN,QTHAIR,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2694121 entries, 0 to 2694120
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   parent_asin  object 
 1   title        object 
 2   price        float64
 3   store        object 
 4   category     object 
dtypes: float64(1), object(4)
memory usage: 102.8+ MB


## Map user/item IDs to integers


In [5]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set(df_reviews["user_id"])
item_ids: set[str] = set(df_reviews["parent_asin"])

user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(item_ids))}

In [6]:
df_reviews['uid'] = df_reviews['user_id'].map(user_map)
df_reviews['iid'] = df_reviews['parent_asin'].map(item_map)
df_items['iid'] = df_items['parent_asin'].map(item_map)


## Filter to beauty category


In [7]:
df_reviews = df_reviews[df_reviews["category"] == BEAUTY_CATEGORY]
print(f"Beauty reviews: {len(df_reviews):,}")

Beauty reviews: 7,300,381


## Filter min user reviews

In [8]:
before_users = df_reviews["uid"].nunique()
user_counts = df_reviews.groupby("uid").size()
valid_users = user_counts[user_counts >= USER_MIN_REVIEWS].index
df_reviews = df_reviews[df_reviews["uid"].isin(valid_users)]
after_users = df_reviews["uid"].nunique()
print(f"Removed {before_users - after_users:,} users with < {USER_MIN_REVIEWS} reviews "
      f"({after_users:,} users remain, {len(df_reviews):,} reviews)")

Removed 4,221,111 users with < 5 reviews (168,377 users remain, 1,585,905 reviews)


## Split and filter train/valid/test 


In [9]:
def _date_to_ms(date_str: str) -> int:
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

timestamps = sorted(df_reviews["timestamp"].values)
train_start_ts = timestamps[0]
train_end_ts = _date_to_ms(TRAIN_END_CUTOFF_DATE)
valid_end_ts = _date_to_ms(VALID_END_CUTOFF_DATE)
print(f"Train start timestamp: {train_start_ts} ({pd.Timestamp(train_start_ts, unit='ms', tz='UTC')})")
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")


Train start timestamp: 1609459264769 (2021-01-01 00:01:04.769000+00:00)
Train end timestamp: 1659312000000 (2022-08-01 00:00:00+00:00)
Valid end timestamp: 1664582400000 (2022-10-01 00:00:00+00:00)


In [10]:
df_train = df_reviews[df_reviews["timestamp"] <= train_end_ts]
df_valid = df_reviews[(df_reviews["timestamp"] > train_end_ts) & (df_reviews["timestamp"] <= valid_end_ts)]
df_test = df_reviews[df_reviews["timestamp"] > valid_end_ts]

print(f"Train reviews: {len(df_train):,}")
print('Train interactions per user', len(df_train) / len(df_train['uid'].unique()))

if MAX_TRAIN_SIZE is not None and len(df_train) > MAX_TRAIN_SIZE:
    df_train = df_train.sample(n=MAX_TRAIN_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled train reviews to {len(df_train):,}")
if MAX_VALID_SIZE is not None and len(df_valid) > MAX_VALID_SIZE:
    df_valid = df_valid.sample(n=MAX_VALID_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled valid reviews to {len(df_valid):,}")
if MAX_TEST_SIZE is not None and len(df_test) > MAX_TEST_SIZE:
    df_test = df_test.sample(n=MAX_TEST_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled test reviews to {len(df_test):,}")

train_users = set(df_train["uid"].unique())
df_valid = df_valid[df_valid["uid"].isin(train_users)]
df_test = df_test[df_test["uid"].isin(train_users)]
df_all = pd.concat([df_train, df_valid, df_test])

print(f"Valid reviews: {len(df_valid):,}")
print(f"Test reviews: {len(df_test):,}")

display(pd.DataFrame({
    "split": ["train", "valid", "test"],
    "reviews": [len(df_train), len(df_valid), len(df_test)],
    "users": [df_train["uid"].nunique(), df_valid["uid"].nunique(), df_test["uid"].nunique()],
    "items": [df_train["iid"].nunique(), df_valid["iid"].nunique(), df_test["iid"].nunique()],
}))

Train reviews: 1,125,023
Train interactions per user 7.18924255688971
Valid reviews: 139,338
Test reviews: 197,706


,split,reviews,users,items
0,train,1125023,156487,215358
1,valid,139338,47209,52826
2,test,197706,55594,66743


## Define cold vs. warm users/items with train data


In [11]:
df_train_users = df_train.groupby("uid").size().reset_index(name="num_train")
cold_user_ids: set[int] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["uid"])
warm_user_ids: set[int] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["uid"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids):,}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids):,}")


Warm users (>= 10 reviews): 20,100
Cold users (< 10 reviews): 136,387


In [12]:
## Define cold vs. warm items with train data
df_train_items = df_train.groupby("iid").size().reset_index(name="num_train")
cold_item_ids: set[int] = set(df_train_items[df_train_items["num_train"] < WARM_ITEM_MIN_REVIEWS]["iid"])
warm_item_ids: set[int] = set(df_train_items[df_train_items["num_train"] >= WARM_ITEM_MIN_REVIEWS]["iid"])
print(f"Warm items (>= {WARM_ITEM_MIN_REVIEWS} train reviews): {len(warm_item_ids):,}")
print(f"Cold items (< {WARM_ITEM_MIN_REVIEWS} train reviews): {len(cold_item_ids):,}")


Warm items (>= 5 train reviews): 50,483
Cold items (< 5 train reviews): 164,875


## Write atomic files


In [13]:
def get_user_category(uid: int) -> int:
    if uid in warm_user_ids:
        return 0  # Warm user
    return 1      # Cold user

def write_user_file(path: Path, uids: set[int]) -> None:
    rows = [(uid, get_user_category(uid)) for uid in uids]
    df_out = pd.DataFrame(rows, columns=["user_id:token", "cold:token"])
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_item_file(path: Path, df_items_subset: pd.DataFrame) -> None:
    df_out = df_items_subset[["iid", "title", "store", "price"]].copy()
    df_out["title"] = df_out["title"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["store"] = df_out["store"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["price"] = pd.to_numeric(df_out["price"], errors="coerce").fillna("")
    df_out["cold"] = df_out["iid"].apply(lambda iid: 1 if iid not in warm_item_ids else 0)
    df_out.columns = ["item_id:token", "title:token", "store:token", "price:float", "cold:token"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_inter_file(path: Path, df: pd.DataFrame) -> None:
    df_out = df[["uid", "iid", "rating", "timestamp"]].copy()
    df_out.columns = ["user_id:token", "item_id:token", "rating:float", "timestamp:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")


In [14]:
dataset_prefix = Path(DATA_DIR) / "beauty" / "beauty"
dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(dataset_prefix.with_suffix(".train.inter"), df_train)
write_inter_file(dataset_prefix.with_suffix(".valid.inter"), df_valid)
write_inter_file(dataset_prefix.with_suffix(".test.inter"), df_test)
write_user_file(dataset_prefix.with_suffix(".user"), set(df_all['uid'].unique()))
write_item_file(dataset_prefix.with_suffix(".item"), df_items[df_items['iid'].isin(df_all['iid'].unique())])


Wrote ../data/beauty/beauty.train.inter (1,125,023 rows)
Wrote ../data/beauty/beauty.valid.inter (139,338 rows)
Wrote ../data/beauty/beauty.test.inter (197,706 rows)
Wrote ../data/beauty/beauty.user (156,487 rows)
Wrote ../data/beauty/beauty.item (250,852 rows)
